# Lesson 19: Programmatic ML Pipelines, MLflow Tracking & JSON Scoring (`score.py`)

In this notebook, you will:
1. Build a Scikit-Learn `Pipeline` + `ColumnTransformer` for the Heart Failure dataset.
2. Tune hyperparameters with `GridSearchCV` and log metrics & artifacts in **MLflow**.
3. Test a production JSON scoring handler (`run(raw_json)`).

In [ ]:
!pip install -q scikit-learn mlflow pandas numpy joblib
import json
import joblib
import mlflow
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
rng = np.random.default_rng(19)
n = 299
age = rng.integers(40, 95, size=n)
ef = rng.integers(14, 80, size=n)
sc = np.round(rng.lognormal(0.2, 0.5, size=n), 2)
time_days = rng.integers(4, 285, size=n)
logit = 0.04 * (age - 60) - 0.05 * (ef - 38) + 0.7 * (sc - 1.3) - 0.02 * (time_days - 130)
y = (rng.random(n) < (1 / (1 + np.exp(-logit)))).astype(int)
X = pd.DataFrame({"age": age, "ejection_fraction": ef, "serum_creatinine": sc, "time": time_days})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

mlflow.set_experiment("heart-failure-sdk-pipeline")
with mlflow.start_run():
    pipe = Pipeline([
        ("prep", ColumnTransformer([("num", StandardScaler(), ["age", "ejection_fraction", "serum_creatinine", "time"])])), 
        ("clf", RandomForestClassifier(random_state=42))
    ])
    search = GridSearchCV(pipe, {"clf__n_estimators": [50, 100], "clf__max_depth": [3, 5]}, scoring="roc_auc", cv=3)
    search.fit(X_train, y_train)
    mlflow.log_params(search.best_params_)
    mlflow.log_metric("cv_roc_auc", search.best_score_)
    joblib.dump(search.best_estimator_, "heart_failure_pipeline.joblib")
    print("Best CV ROC-AUC:", round(search.best_score_, 3))

# Inference endpoint score.py pattern
loaded_model = joblib.load("heart_failure_pipeline.joblib")
payload = json.dumps({"data": [{"age": 60, "ejection_fraction": 38, "serum_creatinine": 1.40, "time": 130}]})
rows = pd.DataFrame(json.loads(payload)["data"])
preds = [bool(p) for p in loaded_model.predict(rows)]
print("Endpoint JSON response:", json.dumps({"result": preds}))